In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import netCDF4 as nc
import os
import pickle


In [2]:
import sys
sys.path.append("/home/z5297792/ESP_zonodo")
from functions import doppio, out_core_param_fit

sys.path.append('/home/z5297792/UNSW-MRes/MRes/SEACOFS_dataset') 
from clim_functions import doppio_pipeliner, find_directional_radii


In [3]:
fname = f'/srv/scratch/z3533156/26year_BRAN2020/outer_avg_01461.nc'

dataset = nc.Dataset(fname)

lon_rho = np.transpose(dataset.variables['lon_rho'], axes=(1, 0))
lat_rho = np.transpose(dataset.variables['lat_rho'], axes=(1, 0))
mask_rho = np.transpose(dataset.variables['mask_rho'], axes=(1, 0))
h = np.transpose(dataset.variables['h'], axes=(1, 0))
# f = np.transpose(dataset.variables['f'], axes=(1, 0))
angle = dataset.variables['angle'][0, 0]
z_r = np.load('/srv/scratch/z5297792/z_r.npy')
z_r = np.transpose(z_r, (1, 2, 0))

def distance(lat1, lon1, lat2, lon2):
    EARTH_RADIUS = 6357
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return EARTH_RADIUS * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

i_mid, j_mid = lon_rho.shape[0] // 2, lon_rho.shape[1] // 2

dx = distance(lat_rho[:-1, j_mid], lon_rho[:-1, j_mid],
              lat_rho[1:, j_mid], lon_rho[1:, j_mid])
dy = distance(lat_rho[i_mid, :-1], lon_rho[i_mid, :-1],
              lat_rho[i_mid, 1:], lon_rho[i_mid, 1:])

x_grid = np.insert(np.cumsum(dx), 0, 0)
y_grid = np.insert(np.cumsum(dy), 0, 0)
X_grid, Y_grid = np.meshgrid(x_grid, y_grid, indexing='ij')


In [4]:
def rotate_uv(u, v, angle):
    u = np.where(np.abs(u) > 1e30, np.nan, u).astype(float)
    v = np.where(np.abs(v) > 1e30, np.nan, v).astype(float)
    u_rot = v * np.sin(angle) + u * np.cos(angle)
    v_rot = v * np.cos(angle) - u * np.sin(angle)

    return u_rot, v_rot
    

In [5]:
df_ameda = pd.read_pickle('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/AMEDA/df_ameda_renamed.pkl')
df_ameda


,Eddy,Day,Cyc,lon,lat,ic,jc,xc,yc,w,...,large1,split,split2,merge,merge2,shapes2,interaction,interaction2,area,deta
0,1,1462,AE,159.538447,-33.372496,256,186,895.838095,919.548973,0.00001,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46653.514371,-0.000011
1,1,1463,AE,159.686966,-33.160751,257,192,903.002279,946.853133,0.000019,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4313.383226,-0.000164
2,1,1464,AE,159.649836,-33.118402,256,192,898.424872,950.150915,0.000019,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4336.799061,-0.00007
3,1,1465,AE,159.686966,-33.118402,257,193,901.765461,951.353349,0.000019,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6480.379311,-0.000047
4,1,1466,AE,159.575577,-32.949006,254,196,886.770848,965.738198,0.00002,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6101.676103,-0.000401
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104565,3519,10643,CE,154.897235,-28.841152,125,252,327.112431,1244.142958,-0.000014,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1147.953129,0.000186
104566,3519,10647,CE,154.860106,-29.052897,127,247,331.141370,1220.843200,-0.000011,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8038.013328,0.000109
104567,3519,10648,CE,154.897235,-29.222293,130,244,340.502393,1204.416117,-0.000011,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9866.414298,0.000095
104568,3519,10649,CE,154.37742,-28.417662,100,257,264.105244,1270.945276,-0.00001,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5581.424065,0.000003


In [6]:
def transect_indexer(ic, jc, X, Y, r=30.0):
    x = np.asarray(X[:, 0])
    y = np.asarray(Y[0, :])

    x0, y0 = x[ic], y[jc]

    i0 = np.searchsorted(x, x0 - r, side='left')
    i1 = np.searchsorted(x, x0 + r, side='right')
    j0 = np.searchsorted(y, y0 - r, side='left')
    j1 = np.searchsorted(y, y0 + r, side='right')

    ii = np.arange(i0, i1)
    jj = np.arange(j0, j1)

    return x[ii], np.full(ii.size, y0), np.full(jj.size, x0), y[jj], ii, jj


def nearest_ij(xc, yc, X, Y):
    x = np.asarray(X[:, 0])
    y = np.asarray(Y[0, :])

    ic = np.abs(x - xc).argmin()
    jc = np.abs(y - yc).argmin()

    return int(ic), int(jc)


def interp_3d_to_reference_depths(var3d, z3d, target_depths):
    target_depths = np.asarray(target_depths)
    nz = target_depths.size

    out = np.full(var3d.shape[:2] + (nz,), np.nan)

    for i in range(var3d.shape[0]):
        for j in range(var3d.shape[1]):
            out[i, j, :] = np.interp(
                target_depths,
                z3d[i, j, :],
                var3d[i, j, :],
                left=np.nan,
                right=np.nan
            )

    return out


def local_eddy_arrays(X, Y, u2d, v2d, xc, yc, Q, rho_max):
    local = (
        (X >= xc - rho_max)
        & (X <= xc + rho_max)
        & (Y >= yc - rho_max)
        & (Y <= yc + rho_max)
    )

    Xloc = X[local]
    Yloc = Y[local]
    uloc = u2d[local]
    vloc = v2d[local]

    dx = Xloc - xc
    dy = Yloc - yc

    rho2 = (
        Q[0, 0] * dx**2
        + 2 * Q[0, 1] * dx * dy
        + Q[1, 1] * dy**2
    )

    rho = np.sqrt(np.where(rho2 >= 0, rho2, np.nan))

    return Xloc, Yloc, uloc, vloc, rho


def vert_doppio_dataset(
    dic, df_ameda, fname, X_grid, Y_grid, angle, z_r, 
    r=30.0,
    max_jump=100,
    max_depth_levels=None,
    rho_max=200,
    rho_min=30,
    target_depths=None
):
    z_r = np.abs(z_r)

    df_file = df_ameda.loc[df_ameda.fname.eq(fname)]

    if df_file.empty:
        return dic

    with nc.Dataset(fname) as ds:

        ocean_time = ds['ocean_time'][:] / 86400
        time_lookup = {int(d): i for i, d in enumerate(ocean_time)}

        for day, df_day in df_file.groupby('Day', sort=False):

            t = time_lookup.get(int(day))
            if t is None:
                continue

            u3d = ds['u_eastward'][t, :, :, :].T.astype(float)
            u3d = np.flip(u3d, axis=2)
            v3d = ds['v_northward'][t, :, :, :].T.astype(float)
            v3d = np.flip(v3d, axis=2)

            u3d[np.abs(u3d) > 1e30] = np.nan
            v3d[np.abs(v3d) > 1e30] = np.nan

            # nz = u3d.shape[-1]
            nz = len(target_depths)
            if max_depth_levels is not None:
                nz = min(nz, max_depth_levels)

            u_depth = interp_3d_to_reference_depths(
                u3d, z_r, target_depths
            )
            
            v_depth = interp_3d_to_reference_depths(
                v3d, z_r, target_depths
            )

            for row in df_day.itertuples():

                eddy_key = f'Eddy{row.Eddy}'
                day_key = f'Day{int(row.Day)}'

                dic.setdefault(eddy_key, {})

                out = []

                xc_prev = row.xc
                yc_prev = row.yc
                w_surf = row.w

                ic = int(row.ic)
                jc = int(row.jc)
                # print(eddy_key)
                for k in range(nz):
                    # print(f'k = {k}')
            
                    if (
                        (xc_prev < r)
                        or (xc_prev > X_grid.max() - r)
                        or (yc_prev < r)
                        or (yc_prev > Y_grid.max() - r)
                    ):
                        # print(1)
                        break

                    target_depth = target_depths[k]

                    u2d = u_depth[:, :, k]
                    v2d = v_depth[:, :, k]

                    x1, y1, x2, y2, ii, jj = transect_indexer(
                        ic, jc, X_grid, Y_grid, r=r
                    )

                    u1 = u2d[ii, jc]
                    v1 = v2d[ii, jc]
                    u2 = u2d[ic, jj]
                    v2 = v2d[ic, jj]

                    u1, v1 = rotate_uv(u1, v1, angle)
                    u2, v2 = rotate_uv(u2, v2, angle)

                    if any(np.all(np.isnan(a)) for a in [u1, v1, u2, v2]):
                        # print(2)
                        break

                    try:
                        xc, yc, w, Q, Omega0 = doppio(
                            x1, y1, u1, v1,
                            x2, y2, u2, v2
                        )
                    except Exception:
                        print(3)
                        break

                    if (
                        np.sign(w) != np.sign(w_surf)
                        or np.hypot(xc - xc_prev, yc - yc_prev) > max_jump
                    ):
                        # print(4)
                        break

                    ic, jc = nearest_ij(xc, yc, X_grid, Y_grid)

                    u2d_rot, v2d_rot = rotate_uv(u2d, v2d, angle)

                    Xloc, Yloc, uloc, vloc, rho = local_eddy_arrays(
                        X_grid, Y_grid,
                        u2d_rot, v2d_rot,
                        xc, yc, Q,
                        rho_max=rho_max
                    )

                    mask0 = rho < rho_max

                    if mask0.sum() < 10:
                        break

                    radii = find_directional_radii(
                        u2d_rot, v2d_rot, X_grid, Y_grid, xc, yc, Q
                    )

                    R = np.nanmean([
                        radii['up'],
                        radii['right'],
                        radii['down'],
                        radii['left'],
                    ])

                    if not np.isfinite(R):
                        # print(5)
                        break

                    rho_lim = max(min(R * 1.75, rho_max), rho_min)
                    mask = rho < rho_lim

                    if mask.sum() < 10:
                        break

                    try:
                        Rc, psi0, Omega = out_core_param_fit(
                            Xloc[mask],
                            Yloc[mask],
                            uloc[mask],
                            vloc[mask],
                            xc, yc, Q,
                            Omega0=Omega0
                        )
                    except Exception:
                        break

                    out.append({
                        'z': k + 1,
                        'Depth': target_depth,
                        'xc': xc,
                        'yc': yc,
                        'ic': ic,
                        'jc': jc,
                        'w': w*1e-3,
                        'Q': Q,
                        'Omega0': Omega0*1e-3,
                        'Omega': Omega*1e-3,
                        'Rc': Rc,
                        'psi0': psi0,
                        'R': R
                    })

                    xc_prev = xc
                    yc_prev = yc

                dic[eddy_key][day_key] = pd.DataFrame(out)

    return dic
    

In [7]:
# save_file = '/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/AMEDA/dic_vert_doppio_AMEDA_para1.pkl'

# fnames = np.sort(df_ameda['fname'].unique())

# # split into 4 roughly equal chunks
# fname_chunks = np.array_split(fnames, 4)

# fname_chunks[0][-1], fname_chunks[1][-1], fname_chunks[2][-1], fname_chunks[3][-1]


('/srv/scratch/z3533156/26year_BRAN2020/outer_avg_03741.nc',
 '/srv/scratch/z3533156/26year_BRAN2020/outer_avg_06051.nc',
 '/srv/scratch/z3533156/26year_BRAN2020/outer_avg_08361.nc',
 '/srv/scratch/z3533156/26year_BRAN2020/outer_avg_10641.nc')

In [9]:
save_file = '/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/AMEDA/dic_vert_doppio_AMEDA_para1.pkl'

fnames = np.sort(df_ameda['fname'].unique())

# split into 4 roughly equal chunks
fname_chunks = np.array_split(fnames, 4)

# choose which worker this script is
worker_id = 0   # 0,1,2,3
fnames = fname_chunks[worker_id]

target_depths = np.abs(z_r[150,150,1:]) 

if os.path.exists(save_file):
    with open(save_file, 'rb') as f:
        dic = pickle.load(f)

    completed = []

    for eddy_key, eddy_dic in dic.items():
        eddy = int(eddy_key[4:])

        for day_key in eddy_dic.keys():
            day = int(day_key[3:])
            completed.append((eddy, day))

    df_completed = pd.DataFrame(completed, columns=['Eddy', 'Day'])

    df_completed = df_completed.merge(
        df_ameda[['Eddy', 'Day', 'fname']],
        on=['Eddy', 'Day'],
        how='left'
    )

    df_completed['fnumber'] = df_completed['fname'].str[48:53].astype(int)

    last_fnumber = df_completed['fnumber'].max()

    print(f'Resuming from fnumber >= {last_fnumber}')

    fnames = [
        f for f in fnames
        if int(f[48:53]) >= last_fnumber
    ]

else:
    dic = {}
    last_fnumber = None
    print('Starting from scratch')

for fname in fnames:

    dic = vert_doppio_dataset(
        dic,
        df_ameda,
        fname,
        X_grid,
        Y_grid,
        angle,
        z_r,
        r=30,
        max_depth_levels=None,
        target_depths=target_depths
    )

    with open(save_file, 'wb') as f:
        pickle.dump(dic, f)

    print(f'Processed {fname}')

    

Resuming from fnumber >= 3741


KeyboardInterrupt: 